# 03 · The deployed risk model on the 100 m grid

**Question:** how well does the platform's deployed risk model rank and allocate forest loss on the platform's 100 m grid, against Verra's benchmark? And is the way the platform validates it sound?

The deployed model is the methodology document's adopted configuration (*Methodology_Deforestation_Guinea_v2*, §7.7), run on the platform:

| | Methodology document | Here |
|---|---|---|
| Grid | 250 m, one value per cell | The platform's 100 m grid; forest and loss from the 30 m shares (notebook 01) |
| Cells the model learns from | Whole cells | The platform's half-cell rule (`HansenShares.display_arrays`; notebook 04) |
| Predictors | Eleven (terrain, access, climate, population), distance to prior loss, palm-oil price; yearly layers read at their latest year | The same, without the palm-oil price (notebook 09 tests it: it adds nothing) |
| Windows | Three; the latest calibrated on 2015–2018 and tested on 2019–2022 | Three: 2014–2016 → 2017–2019, 2017–2019 → 2020–2022, 2019–2021 → 2022–2024 |
| Scores | AUC over cells: 0.564, 0.660 and 0.689 | AUC over the 30 m forest pixels, and Verra's two tests |

**Verra's VT0007 tests** (v1.0, 2024), which the methodology document cites but does not report: the model's risk map is cut into 30 classes, each class gets the loss rate it had in the calibration window, and the loss is allocated to the cells by those rates. The test is the median absolute error (MedAE, ha) of the allocated against the observed loss over the 185 blocks of about 100,000 ha wholly in Guinea:

* **fit test:** on the calibration window itself;
* **prediction test:** on the next window, with the rates of the calibration window.

Verra's **benchmark** ranks the forest by its distance from non-forest only. A model qualifies only if it beats the benchmark on both tests. Each cell's forest and loss are in hectares from its shares. Forest without a risk value gets no class and no allocated loss.

**AUC over the 30 m forest pixels** (notebook 04): the chance that a pixel lost in the test window ranks above one that stayed, mean of three seeds. Forest the risk map leaves out counts as lowest risk, as it would on the platform.

The code is `scripts/verra.py`. Results are saved in `outputs/runs/`; set `RERUN = True` to compute them again (about 15 minutes per window).

**Why Verra's tests are here, not in `3_benchmark/`:** they are the scoring standard every model in these notebooks is judged by (the methodology document adopts VT0007, §2). So the deployed model is scored on them first, and every later notebook compares with these numbers. Verra's benchmark is part of that standard: the reference model a candidate must beat to qualify. It is not one of the other tools, which `3_benchmark/` compares.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "scripts"))
import model as MM
import verra as V
import holdout_timing as FX

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
OUT = HERE.parent / "outputs" / "summary"
RERUN = False
runs = [MM.cached(V.split_name(s), lambda s=s: V.run_split(s), rerun=RERUN) for s in MM.SPLITS]
timing = [MM.cached(FX.split_name(s), lambda s=s: FX.predictor_timing(s)) for s in MM.SPLITS]
MODELS = ["benchmark", "deployed_half"]

## 1 · The deployed model against Verra's benchmark

In [2]:
rows = []
for run in runs:
    bench = run["models"]["benchmark"]
    for key in MODELS:
        r = run["models"][key]
        rows.append({"Window": run["split"], "Model": V.LABELS[key],
                     "AUC (30 m pixels)": np.mean(r["aucs"]),
                     "Fit MedAE (ha)": r["fit_medae_ha"], "Prediction MedAE (ha)": r["prediction_medae_ha"],
                     "Beats the benchmark on both tests": "–" if key == "benchmark" else
                         ("yes" if r["fit_medae_ha"] < bench["fit_medae_ha"] and r["prediction_medae_ha"] < bench["prediction_medae_ha"] else "no"),
                     "Next loss in forest without a risk value (%)": 100 * r["loss_without_class"]})
baseline = pd.DataFrame(rows)
baseline.round(4).to_csv(OUT / "03_deployed_baseline.csv", index=False)
mean = baseline.groupby("Model", sort=False)[["AUC (30 m pixels)", "Fit MedAE (ha)", "Prediction MedAE (ha)",
                                              "Next loss in forest without a risk value (%)"]].mean()
mean.index = [f"{m}, mean of 3 windows" for m in mean.index]
display(baseline.set_index(["Window", "Model"]).round(3))
mean.round(3)

AUC (30 m pixels)  \
Window                Model                                                           
2014–2016 → 2017–2019 VT0007 benchmark (distance to edge)                     0.694   
                      Deployed model, half-cell masks (platform)              0.717   
2017–2019 → 2020–2022 VT0007 benchmark (distance to edge)                     0.673   
                      Deployed model, half-cell masks (platform)              0.681   
2019–2021 → 2022–2024 VT0007 benchmark (distance to edge)                     0.638   
                      Deployed model, half-cell masks (platform)              0.631   

                                                                  Fit MedAE (ha)  \
Window                Model                                                        
2014–2016 → 2017–2019 VT0007 benchmark (distance to edge)                251.760   
                      Deployed model, half-cell masks (platform)         129.938   
2017–2019 → 2020–2022 VT0007 benchmark (distance to edge)                385.399   
                      Deployed model, half-cell masks (platform)         185.083   
2019–2021 → 2022–2024 VT0007 benchmark (distance to edge)                422.414   
                      Deployed model, half-cell masks (platform)         156.374   

                                                                  Prediction MedAE (ha)  \
Window                Model                                                               
2014–2016 → 2017–2019 VT0007 benchmark (distance to edge)                       445.637   
                      Deployed model, half-cell masks (platform)                278.241   
2017–2019 → 2020–2022 VT0007 benchmark (distance to edge)                       376.290   
                      Deployed model, half-cell masks (platform)                216.335   
2019–2021 → 2022–2024 VT0007 benchmark (distance to edge)                       297.709   
                      Deployed model, half-cell masks (platform)                213.474   

                                                                 Beats the benchmark on both tests  \
Window                Model                                                                          
2014–2016 → 2017–2019 VT0007 benchmark (distance to edge)                                        –   
                      Deployed model, half-cell masks (platform)                               yes   
2017–2019 → 2020–2022 VT0007 benchmark (distance to edge)                                        –   
                      Deployed model, half-cell masks (platform)                               yes   
2019–2021 → 2022–2024 VT0007 benchmark (distance to edge)                                        –   
                      Deployed model, half-cell masks (platform)                               yes   

                                                                  Next loss in forest without a risk value (%)  
Window                Model                                                                                     
2014–2016 → 2017–2019 VT0007 benchmark (distance to edge)                                                0.000  
                      Deployed model, half-cell masks (platform)                                         9.544  
2017–2019 → 2020–2022 VT0007 benchmark (distance to edge)                                                0.001  
                      Deployed model, half-cell masks (platform)                                        14.883  
2019–2021 → 2022–2024 VT0007 benchmark (distance to edge)                                                0.002  
                      Deployed model, half-cell masks (platform)                                        19.810

,AUC (30 m pixels),Fit MedAE (ha),Prediction MedAE (ha),Next loss in forest without a risk value (%)
"VT0007 benchmark (distance to edge), mean of 3 windows",0.668,353.191,373.212,0.001
"Deployed model, half-cell masks (platform), mean of 3 windows",0.676,157.131,236.017,14.746


**Reading:**

* **The deployed model beats Verra's benchmark on both tests in every window:** fit errors of 130–185 ha against 252–422, prediction errors of 213–278 ha against 298–446. So it qualifies under VT0007, which the methodology document cites but did not test.
* **It ranks the loss only a little better than the benchmark** (AUC 0.676 against 0.668 on average), and not at all in the last window (0.631 against 0.638).
* **The reason is the half-cell masks.** The risk map covers only cells at least half forest, so the next window's loss outside them gets no risk value: 10 % of it in the first window, 15 % in the second, 20 % in the third. The share grows as the forest thins (notebook 04).
* **These AUCs are not comparable with the methodology document's 0.56–0.69.** Those are over 250 m cells, with one value per cell, on other windows. Section 3 gives the score the platform's own validation would give here.

## 2 · What the deployed model relies on

Random-forest importance, first seed, mean of the three windows.

In [3]:
imp = pd.DataFrame([run["models"]["deployed_half"]["importance"] for run in runs]).mean().sort_values(ascending=False)
imp.rename("Importance").to_frame().round(3)

,Importance
Land-surface temperature,0.160
Distance to prior loss,0.118
Precipitation,0.113
Elevation,0.096
Soil moisture,0.090
Slope,0.076
Distance to railways,0.072
Points of interest,0.068
Distance to waterways,0.063
Aspect,0.058


**Reading:** the deployed model relies most on land-surface temperature, then the distance to prior loss and rainfall. Temperature, rainfall and soil moisture are read at 2024, after the loss they predict, and clearing itself warms the ground (`studies/forest_loss_drivers.ipynb`, section 2). Section 3 measures what this does to its score.

## 3 · The platform's hold-out reads predictors at their latest year

The platform's hold-out reads each yearly predictor (rainfall, land temperature, soil moisture, night lights, population) at its latest year, 2024 (2020 for population), whatever window it tests. So a model tested on the loss of 2017–2019 sees the climate and population of 2024. Here the deployed model is trained and tested both ways, three seeds each, and scored as the platform scores it: its own half-forest cells, lost or kept by the half-cell rule. The code is `scripts/holdout_timing.py`.

In [4]:
rows = []
for t in timing:
    for key, label in FX.TIMINGS.items():
        rows.append({"Test window": t["split"].split(" → ")[1], "Predictors read at": label,
                     "AUC, the platform's own score": np.mean(t["timings"][key]["aucs_own"])})
timing_table = pd.DataFrame(rows)
timing_table.round(4).to_csv(OUT / "03_predictor_timing.csv", index=False)
wide = timing_table.pivot(index="Test window", columns="Predictors read at", values="AUC, the platform's own score")
wide = wide[list(FX.TIMINGS.values())]
wide["Latest year minus window start"] = wide.iloc[:, 0] - wide.iloc[:, 1]
print("Deployed model, mean of three seeds")
wide.round(4)

Deployed model, mean of three seeds


Predictors read at,Latest year (platform),Window start,Latest year minus window start
Test window,,,
2017–2019,0.7611,0.7530,0.0081
2020–2022,0.7580,0.7434,0.0147
2022–2024,0.7329,0.7276,0.0053


**Reading:**

* **Reading the yearly predictors at their latest year raises the deployed model's score in every window:** +0.008, +0.015 and +0.005 (mean +0.009). That is above the run-to-run noise (0.003). The model is tested with climate and population measured after the loss it predicts.
* **For the improved model it makes no difference** (notebook 05: +0.0001), because its loss history carries most of the signal.
* **So the platform's validation score overstates the deployed model by about 0.01 AUC.**
* **The fix costs nothing:** in the hold-out, read every predictor at the calibration window's start. For the forecast, the latest year is the window's start, so the risk map itself does not change.

## 4 · What this means

* **The deployed model qualifies under Verra's VT0007:** it beats the benchmark on both tests in every window. The methodology document reports only AUC. The platform's validation should add Verra's two tests, with forest and loss in hectares from the shares.
* **Three problems in how it is trained and validated**, each with its fix in the next notebooks:
  1. **The half-cell masks leave 10–20 % of the next loss without a risk value** (section 1), and the share grows as the forest thins. Fix: train on the shares and score every forest cell (notebook 04).
  2. **The hold-out reads yearly predictors at their latest year** (section 3), which overstates its score by about 0.01 AUC. Fix: read every predictor at the window's start; the forecast does not change.
  3. **It leans on predictors measured after the loss** (temperature, rainfall and soil moisture at 2024), and its only loss history is the distance to prior loss. Notebook 05 tests the predictors.
* **Next, notebook 04:** which cells the model should learn from.